# 01: PyTorch models

This notebook defines the three neural architectures used to study the same cascade prediction problem:

1. **MLP baseline:** converts the whole graph in a single long vector;
2. **GNN predictor:** uses graph edges to propagate information locally, each node receives information from its neighbours;
3. **Hybrid GNN-Transformer:** combines local graph convolutions with a global structural attention.

All three models return the same outputs. This is useful because the training notebook can use one common loss function.
### Tensor notation used below

- `B`: number of graphs in a batch;
- `N`: number of nodes;
- `F`: number of input features per node;
- `H`: hidden representation size.

A node tensor therefore usually has shape `(B, N, F)` or `(B, N, H)`.

In [1]:
import math
import random

import numpy as np
import pandas as pd
import torch
from torch import nn


## Reproducibility and device

PyTorch uses random numbers when it initializes neural-network weights. Setting the seeds allows for reproducibility.

The `device` object tells PyTorch whether tensors and models should be placed on a GPU or on the CPU. (useful for later project implementation)

In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device:", device)


Device: cpu


## Shared prediction heads

The GNN and the hybrid model first produce one hidden vector for every node. The following heads converts those vectors to a more suitable format that allows to efficiently calculate the three predictions required by the project:

- one failure **activation** for every node;
- one predicted future load for every node;
- one graph-level estimate of the final failed fraction.

The activation is the raw output before applying a sigmoid. Keeping raw logits is numerically more stable when using `binary_cross_entropy_with_logits`.

In [3]:
def masked_mean(node_embeddings, mask):
    # mask has shape (B, N). The extra dimension makes it compatible
    # with node_embeddings, which has shape (B, N, H).
    mask_as_float = mask.unsqueeze(-1).to(node_embeddings.dtype)

    # Padded nodes are multiplied by zero and therefore do not contribute.
    summed_embeddings = (node_embeddings * mask_as_float).sum(dim=1)

    # Count the real nodes in each graph. clamp_min avoids division by zero (advanced protection suggested by AI).
    number_of_nodes = mask_as_float.sum(dim=1).clamp_min(1.0)

    return summed_embeddings / number_of_nodes  # basically we are computing a global embedding of the graph via the average


class PredictionHeads(nn.Module):
    def __init__(self, hidden_dim, dropout):
        super().__init__()

        # Each head is a small feed-forward network applied independently (a projection head like in constrstive learning).
        self.node_failure_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

        self.node_load_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

        self.graph_risk_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, node_embeddings, mask):
        # The final dimension is 1, so squeeze removes only that dimension.
        node_activation = self.node_failure_head(node_embeddings).squeeze(-1)
        load_prediction = self.node_load_head(node_embeddings).squeeze(-1)

        # A graph vector is obtained by averaging only its real nodes.
        graph_embedding = masked_mean(node_embeddings, mask)
        graph_activation = self.graph_risk_head(graph_embedding).squeeze(-1)

        # The graph target is a fraction in [0, 1], so a sigmoid is applied here.
        graph_prediction = torch.sigmoid(graph_activation)

        return {
            "node_activation": node_activation,
            "load_pred": load_prediction,
            "graph_pred": graph_prediction,
        }


## MLP baseline

The MLP is intentionally simple. It receives the node-feature matrix and adjacency matrix, flattens both, and treats the complete graph as one vector.

This removes the explicit notion of neighbourhood. It is therefore a useful non-graph baseline; as seen it requires every graph to have the same number of nodes.

In [4]:
class MLPBaseline(nn.Module):
    def __init__(self, num_nodes, feature_dim, hidden_dim, dropout):
        super().__init__()

        self.num_nodes = num_nodes

        # Node features contribute N * F numbers.
        # The dense adjacency matrix contributes N * N numbers.
        input_dim = num_nodes * feature_dim + num_nodes * num_nodes

        self.backbone = nn.Sequential(
            nn.Linear(input_dim, hidden_dim * 2),   # hidden dim*2 is a design choice related to the dimension of the input
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # Separate linear layers produce the three requested outputs, we basically replicate the previous heads in a more mlp manner for this particular model.
        """
        The mlp learns the internal representation of the graph or its nodes (the hidden embeddings).
        The heads then convert that representation into the quantities you actually want to predict
        """
        self.node_failure_head = nn.Linear(hidden_dim * 2, num_nodes)
        self.node_load_head = nn.Linear(hidden_dim * 2, num_nodes)
        self.graph_risk_head = nn.Linear(hidden_dim * 2, 1)

    def forward(self, batch):
        x = batch["x"]
        adjacency = batch["adj"]
        mask = batch["mask"]

        # The MLP cannot naturally ignore padding, so all graphs must be full size.
        if x.shape[1] != self.num_nodes or not bool(mask.all()):
            raise ValueError("MLPBaseline requires fixed-size, unpadded graphs.")

        # flatten(start_dim=1) keeps the batch dimension and joins the others.
        flat_features = x.flatten(start_dim=1)
        flat_adjacency = adjacency.flatten(start_dim=1)

        # Concatenation creates one vector for each graph in the batch.
        # The idea is that we are feeding a one dimensional vector to the MLP, that we are here defining.
        model_input = torch.cat([flat_features, flat_adjacency], dim=1)
        hidden = self.backbone(model_input) # We get the hiddens feeding to the architecture (backbone) the inputs

        node_activation = self.node_failure_head(hidden)
        load_prediction = self.node_load_head(hidden)
        graph_activation = self.graph_risk_head(hidden).squeeze(-1)

        return {
            "node_activation": node_activation,
            "load_pred": load_prediction,
            "graph_pred": torch.sigmoid(graph_activation),
        }


## Dense graph convolution

The graph convolution implements the common GCN operation

$$
H' = \hat{D}^{-1/2}\hat{A}\hat{D}^{-1/2}HW,
$$

where `A + I` adds self-connections. In words, every node combines its transformed features with the transformed features of its neighbours.

This steps serves the purpose of transforming the adjacency matrix into a normalized adjacency matrix so that the GNN can aggregate information from neighbouring nodes in a stable and meaningful way.

The implementation uses dense adjacency matrices because each graph contains relatively few nodes.

In [5]:
class DenseGraphConv(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()

        # nn.Linear stores the learnable matrix W and its bias.
        self.linear = nn.Linear(input_dim, output_dim)

    def forward(self, x, adjacency, mask):
        batch_size = x.shape[0]
        num_nodes = x.shape[1]

        # torch.eye creates the identity matrix I used for self-connections.
        identity = torch.eye(
            num_nodes,
            device=x.device,
            dtype=adjacency.dtype,
        )
        identity = identity.unsqueeze(0).expand(batch_size, -1, -1)

        # A pair is valid only when both nodes are real rather than padding.
        valid_rows = mask.unsqueeze(2)
        valid_columns = mask.unsqueeze(1)
        valid_pairs = valid_rows & valid_columns    # logical AND to create a mask

        adjacency_with_self_loops = adjacency + identity
        adjacency_with_self_loops = (
            adjacency_with_self_loops * valid_pairs.to(adjacency.dtype)
        )

        # The degree is the number of incident connections after adding self-loops.
        degree = adjacency_with_self_loops.sum(dim=-1).clamp_min(1.0)
        inverse_sqrt_degree = degree.rsqrt()

        # This is the symmetric normalization D^(-1/2) A D^(-1/2).
        normalized_adjacency = (
            inverse_sqrt_degree.unsqueeze(-1)
            * adjacency_with_self_loops
            * inverse_sqrt_degree.unsqueeze(-2)
        )

        transformed_features = self.linear(x)

        # @ performs batched matrix multiplication:
        # (B, N, N) @ (B, N, H) -> (B, N, H).
        output = normalized_adjacency @ transformed_features

        # Padded node positions are returned to zero.
        return output * mask.unsqueeze(-1).to(output.dtype)


## GNN predictor

`nn.ModuleList` is used because the graph-convolution layers must be registered inside the model. Registration is what allows PyTorch to find their parameters when calling `model.parameters()`.

The residual addition is used from the second layer onward, when the old and new tensors both have hidden dimension `H`.

In [6]:
class GNNPredictor(nn.Module):
    def __init__(self, feature_dim, hidden_dim, num_layers, dropout):
        super().__init__()

        self.layers = nn.ModuleList()
        self.normalizations = nn.ModuleList()

        current_dim = feature_dim
        for _ in range(num_layers):
            self.layers.append(DenseGraphConv(current_dim, hidden_dim))
            self.normalizations.append(nn.LayerNorm(hidden_dim))
            current_dim = hidden_dim

        self.dropout = nn.Dropout(dropout)
        self.heads = PredictionHeads(hidden_dim, dropout)

    def encode(self, batch):
        hidden = batch["x"]
        adjacency = batch["adj"]
        mask = batch["mask"]

        for layer_index, layer in enumerate(self.layers):
            updated = layer(hidden, adjacency, mask)
            updated = torch.relu(updated)

            # A residual connection helps information and gradients travel through multiple layers.
            # It is possible only when shapes match.
            if layer_index > 0:
                updated = updated + hidden

            updated = self.dropout(updated)
            hidden = self.normalizations[layer_index](updated)

            # LayerNorm has a learnable bias, so padding is explicitly zeroed again.
            hidden = hidden * mask.unsqueeze(-1).to(hidden.dtype)

        return hidden

    def forward(self, batch):
        node_embeddings = self.encode(batch)
        return self.heads(node_embeddings, batch["mask"])


## Structural self-attention

The hybrid model first performs local GNN message passing. It then uses self-attention so that every node can directly compare itself with every other node.

The shortest-path distance matrix contributes a learned bias. Nodes separated by different graph distances can therefore receive different attention preferences.

In [7]:
class StructuralSelfAttention(nn.Module):
    def __init__(self, hidden_dim, num_heads, max_distance, dropout):
        super().__init__()

        if hidden_dim % num_heads != 0:
            raise ValueError("hidden_dim must be divisible by num_heads")

        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        self.max_distance = max_distance

        # One projection creates queries, keys and values together.
        self.qkv_projection = nn.Linear(hidden_dim, hidden_dim * 3)
        self.output_projection = nn.Linear(hidden_dim, hidden_dim)

        # The last index is reserved for distances larger than max_distance
        # or for node pairs that are not connected by a finite path.
        self.distance_bias = nn.Embedding(max_distance + 2, num_heads)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, distance, mask):
        batch_size, num_nodes, _ = x.shape

        qkv = self.qkv_projection(x)
        qkv = qkv.reshape(
            batch_size,
            num_nodes,
            3,
            self.num_heads,
            self.head_dim,
        )

        query, key, value = qkv.unbind(dim=2)

        # Move the head dimension before the node dimension:
        # (B, N, heads, head_dim) -> (B, heads, N, head_dim).
        query = query.transpose(1, 2)
        key = key.transpose(1, 2)
        value = value.transpose(1, 2)

        attention_scores = query @ key.transpose(-1, -2)
        attention_scores = attention_scores / math.sqrt(self.head_dim)

        distance_index = distance.clamp(
            min=0,
            max=self.max_distance + 1,
        )
        structural_bias = self.distance_bias(distance_index)
        structural_bias = structural_bias.permute(0, 3, 1, 2)
        attention_scores = attention_scores + structural_bias

        # Padded keys must never receive attention probability.
        very_negative = torch.finfo(attention_scores.dtype).min
        attention_scores = attention_scores.masked_fill(
            ~mask[:, None, None, :],
            very_negative,
        )

        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.dropout(attention_weights)

        attended_values = attention_weights @ value

        # Join all attention heads back into one hidden vector per node.
        attended_values = attended_values.transpose(1, 2)
        attended_values = attended_values.reshape(
            batch_size,
            num_nodes,
            self.hidden_dim,
        )

        output = self.output_projection(attended_values)
        return output * mask.unsqueeze(-1).to(output.dtype)


In [8]:
class StructuralTransformerBlock(nn.Module):
    def __init__(
        self,
        hidden_dim,
        num_heads,
        feed_forward_dim,
        max_distance,
        dropout,
    ):
        super().__init__()

        self.attention = StructuralSelfAttention(
            hidden_dim,
            num_heads,
            max_distance,
            dropout,
        )

        self.norm_after_attention = nn.LayerNorm(hidden_dim)
        self.norm_after_feed_forward = nn.LayerNorm(hidden_dim)

        self.feed_forward = nn.Sequential(
            nn.Linear(hidden_dim, feed_forward_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(feed_forward_dim, hidden_dim),
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, distance, mask):
        attention_output = self.attention(x, distance, mask)

        # Both sub-layers use residual connections followed by LayerNorm.
        x = self.norm_after_attention(
            x + self.dropout(attention_output)
        )

        feed_forward_output = self.feed_forward(x)
        x = self.norm_after_feed_forward(
            x + self.dropout(feed_forward_output)
        )

        return x * mask.unsqueeze(-1).to(x.dtype)


## Hybrid GNN–Transformer

Laplacian positional encodings are projected to the same hidden dimension as the GNN embeddings and then added to them. This gives the attention layers information about the node position in the graph structure.

In [9]:
class HybridGNNTransformer(nn.Module):
    def __init__(
        self,
        feature_dim,
        pe_dim,
        hidden_dim,
        num_gnn_layers,
        transformer_layers,
        transformer_heads,
        feed_forward_dim,
        max_distance,
        dropout,
    ):
        super().__init__()

        self.gnn_layers = nn.ModuleList()
        self.gnn_normalizations = nn.ModuleList()

        current_dim = feature_dim
        for _ in range(num_gnn_layers):
            self.gnn_layers.append(
                DenseGraphConv(current_dim, hidden_dim)
            )
            self.gnn_normalizations.append(
                nn.LayerNorm(hidden_dim)
            )
            current_dim = hidden_dim

        self.pe_projection = nn.Linear(pe_dim, hidden_dim)

        self.transformer_blocks = nn.ModuleList()
        for _ in range(transformer_layers):
            block = StructuralTransformerBlock(
                hidden_dim=hidden_dim,
                num_heads=transformer_heads,
                feed_forward_dim=feed_forward_dim,
                max_distance=max_distance,
                dropout=dropout,
            )
            self.transformer_blocks.append(block)

        self.dropout = nn.Dropout(dropout)
        self.heads = PredictionHeads(hidden_dim, dropout)

    def encode(self, batch):
        hidden = batch["x"]
        adjacency = batch["adj"]
        mask = batch["mask"]

        # First stage: local propagation through the graph.
        for layer_index, layer in enumerate(self.gnn_layers):
            updated = torch.relu(layer(hidden, adjacency, mask))

            if layer_index > 0:
                updated = updated + hidden

            updated = self.dropout(updated)
            hidden = self.gnn_normalizations[layer_index](updated)
            hidden = hidden * mask.unsqueeze(-1).to(hidden.dtype)

        # The positional encoding has shape (B, N, pe_dim).
        positional_information = self.pe_projection(batch["lap_pe"])
        hidden = hidden + positional_information
        hidden = hidden * mask.unsqueeze(-1).to(hidden.dtype)

        # Second stage: global communication through attention.
        for block in self.transformer_blocks:
            hidden = block(hidden, batch["dist"], mask)

        return hidden

    def forward(self, batch):
        node_embeddings = self.encode(batch)
        return self.heads(node_embeddings, batch["mask"])


## Model factory and parameter counting

A factory is a small function that constructs the requested model from the shared configuration. It keeps the training notebook independent from model-specific constructor arguments.

In [10]:
def build_model(model_name, metadata, config):
    model_config = config["model"]

    feature_dim = int(metadata["feature_dim"])
    hidden_dim = int(model_config["hidden_dim"])
    dropout = float(model_config["dropout"])

    if model_name == "mlp":
        return MLPBaseline(
            num_nodes=int(metadata["max_nodes"]),
            feature_dim=feature_dim,
            hidden_dim=hidden_dim,
            dropout=dropout,
        )

    if model_name == "gnn":
        return GNNPredictor(
            feature_dim=feature_dim,
            hidden_dim=hidden_dim,
            num_layers=int(model_config["num_gnn_layers"]),
            dropout=dropout,
        )

    if model_name == "hybrid":
        return HybridGNNTransformer(
            feature_dim=feature_dim,
            pe_dim=int(metadata["lap_pe_dim"]),
            hidden_dim=hidden_dim,
            num_gnn_layers=int(model_config["num_gnn_layers"]),
            transformer_layers=int(model_config["transformer_layers"]),
            transformer_heads=int(model_config["transformer_heads"]),
            feed_forward_dim=int(model_config["ff_dim"]),
            max_distance=int(config["data"]["max_distance"]),
            dropout=dropout,
        )

    raise ValueError(f"Unknown model: {model_name}")


def count_parameters(model):
    total = 0

    for parameter in model.parameters():
        if parameter.requires_grad:
            total += parameter.numel()

    return total


## Synthetic batch: forward and backward check

Before training, we test the models with artificial tensors. This does not measure performance. It checks that:

- the expected tensor shapes are accepted;
- each output has the correct shape;
- the loss creates gradients through PyTorch autograd;
- no output contains `NaN` or infinite values.

In [11]:
BATCH_SIZE = 2
NUM_NODES = 20
FEATURE_DIM = 10
PE_DIM = 4
HIDDEN_DIM = 32

# Random node features: one F-dimensional vector for each node.
dummy_x = torch.randn(
    BATCH_SIZE,
    NUM_NODES,
    FEATURE_DIM,
    device=device,
)

# Start from a random binary adjacency matrix.
dummy_adjacency = torch.randint(
    low=0,
    high=2,
    size=(BATCH_SIZE, NUM_NODES, NUM_NODES),
    device=device,
).float()

# Make the graph undirected by copying every connection in both directions.
dummy_adjacency = torch.maximum(
    dummy_adjacency,
    dummy_adjacency.transpose(1, 2),
)

# Remove self-connections here. The GNN layer will add them internally.
identity = torch.eye(NUM_NODES, device=device).unsqueeze(0)
dummy_adjacency = dummy_adjacency * (1.0 - identity)

# All nodes are valid in this fixed-size synthetic example.
dummy_mask = torch.ones(
    BATCH_SIZE,
    NUM_NODES,
    dtype=torch.bool,
    device=device,
)

dummy_lap_pe = torch.randn(
    BATCH_SIZE,
    NUM_NODES,
    PE_DIM,
    device=device,
)

dummy_distance = torch.randint(
    low=0,
    high=8,
    size=(BATCH_SIZE, NUM_NODES, NUM_NODES),
    device=device,
)

dummy_batch = {
    "x": dummy_x,
    "adj": dummy_adjacency,
    "mask": dummy_mask,
    "lap_pe": dummy_lap_pe,
    "dist": dummy_distance,
}


In [12]:
models_to_test = {
    "mlp": MLPBaseline(
        num_nodes=NUM_NODES,
        feature_dim=FEATURE_DIM,
        hidden_dim=HIDDEN_DIM,
        dropout=0.1,
    ),
    "gnn": GNNPredictor(
        feature_dim=FEATURE_DIM,
        hidden_dim=HIDDEN_DIM,
        num_layers=2,
        dropout=0.1,
    ),
    "hybrid": HybridGNNTransformer(
        feature_dim=FEATURE_DIM,
        pe_dim=PE_DIM,
        hidden_dim=HIDDEN_DIM,
        num_gnn_layers=2,
        transformer_layers=1,
        transformer_heads=4,
        feed_forward_dim=64,
        max_distance=6,
        dropout=0.1,
    ),
}

test_rows = []

for model_name, model in models_to_test.items():
    model = model.to(device)
    model.train()

    # Calling model(...) automatically calls its forward method.
    outputs = model(dummy_batch)

    print(f"\n{model_name}")
    for output_name, output_tensor in outputs.items():
        print(output_name, tuple(output_tensor.shape))

    assert outputs["node_activation"].shape == (
        BATCH_SIZE,
        NUM_NODES,
    )
    assert outputs["load_pred"].shape == (
        BATCH_SIZE,
        NUM_NODES,
    )
    assert outputs["graph_pred"].shape == (BATCH_SIZE,)

    # A simple scalar is sufficient to test autograd.
    test_loss = outputs["node_activation"].mean()
    test_loss = test_loss + outputs["load_pred"].mean()
    test_loss = test_loss + outputs["graph_pred"].mean()

    test_loss.backward()

    has_gradient = False
    for parameter in model.parameters():
        if parameter.requires_grad and parameter.grad is not None:
            has_gradient = True
            break

    assert has_gradient

    for output_tensor in outputs.values():
        assert torch.isfinite(output_tensor).all()

    test_rows.append(
        {
            "model": model_name,
            "trainable_parameters": count_parameters(model),
        }
    )

pd.DataFrame(test_rows)



mlp
node_activation (2, 20)
load_pred (2, 20)
graph_pred (2,)

gnn
node_activation (2, 20)
load_pred (2, 20)
graph_pred (2,)

hybrid
node_activation (2, 20)
load_pred (2, 20)
graph_pred (2,)


,model,trainable_parameters
0,mlp,45289
1,gnn,4803
2,hybrid,13539


The next notebook uses these models inside a complete training loop, and, these functions are stored in `dl_models.py` so that they can be imported by the other notebooks.